# GitSkills Exploration

A playground for poking around the GitSkills SQLite sample. Each section has a short note, then a code cell.
Lines marked `# <- change me` are the ones meant for editing.

- Run with **Shift+Enter**, or **Run All** at the top.
- Run section 1 first. Section 4 needs section 3, and section 6 needs section 4.
- If you restart the kernel, run section 1 again (and 3 → 4 before 6).

## 1. Connect

This opens `data/sample/agent_skills_sample.db` in **read-only** mode, so nothing we run can change the data.
Every other cell talks to the database through this connection, so run this cell first.
It also defines a small helper, `sql("SELECT ...")`, that runs a query and returns a pandas DataFrame (a table).

You normally don't need to change the path. If you get "Database not found", the `.db` file isn't in
`data/sample/` yet (see the README).

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)  # max characters shown per cell in tables; raise it to see longer text

# Works whether VS Code runs the notebook from notebooks/ or from the project root
ROOT = Path.cwd() if (Path.cwd() / "data" / "sample").exists() else Path.cwd().parent
DB_PATH = ROOT / "data" / "sample" / "agent_skills_sample.db"
assert DB_PATH.exists(), f"Database not found at {DB_PATH} (see README)"

# Read-only, so nothing in the database can be changed by accident
conn = sqlite3.connect(f"{DB_PATH.resolve().as_uri()}?mode=ro", uri=True)


# sql("SELECT ...") -> DataFrame. Use ? placeholders for values: sql("... WHERE name = ?", ("pdf",))
def sql(query, params=()):
    return pd.read_sql_query(query, conn, params=params)


print("Connected to", DB_PATH.relative_to(ROOT))

## 2. Database overview

The four tables:

- **`artifacts`**: one row per SKILL.md found on GitHub (a repo + path). Holds the skill's `name`, `description`,
  full text (`content`), folder stats and commit info.
- **`artifact_siblings`**: the other files in a skill's folder (scripts, reference docs, ...). Linked to
  `artifacts` by `repo_full_name` + `artifact_path` = `artifacts.repo_full_name` + `path`.
- **`repos`**: GitHub info for each repository (stars, forks, language, license, ...). Linked by
  `repos.full_name` = `artifacts.repo_full_name`.
- **`mining_runs`**: a log of the GitHub searches used to collect the data. Mostly bookkeeping.

The second table shows `skill_occurrences` (every SKILL.md file found) vs `distinct_skill_files` (unique contents).
There are many more occurrences because the same skill gets copied between repos: forks, skill collections, and
people vendoring popular skills. Copies share a `file_sha`, and only one "representative" copy
(`dedup_primary = 1`) stores the text and supporting files.

The last output lists the columns of one table. Set `table` to another name to see its columns.

In [ ]:
tables = sql("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name")
tables["rows"] = [sql(f'SELECT COUNT(*) AS n FROM "{t}"')["n"][0] for t in tables["name"]]
display(tables)

display(sql("""
SELECT COUNT(*)                       AS skill_occurrences,
       SUM(dedup_primary)             AS distinct_skill_files,
       COUNT(DISTINCT repo_full_name) AS repos
FROM artifacts
"""))

# Columns of any table
table = "artifacts"  # <- change me: "artifacts", "artifact_siblings", "repos", "mining_runs"
sql(f'PRAGMA table_info("{table}")')[["name", "type"]]

## 3. Browse skills

Shows `n` distinct skills (representatives only, so each unique SKILL.md appears once), sorted by repo stars.
Change `n` to show more or fewer rows. The result is saved as `skills`, which section 4 uses.

- `repo`, `path`: where the SKILL.md lives on GitHub
- `name`, `description`: from the skill's frontmatter (the block at the top of the file). Can be empty.
- `stars`: the repo's GitHub stars
- `files`: number of other files in the skill's folder
- `has_scripts`, `has_references`: 1 if the folder has script files (.py, .sh, ...) or docs (.md, .txt, ...)
- `copies`: how many places in the dataset have this exact same SKILL.md (1 means it's unique)

To explore differently, uncomment one of the `-- AND ...` filter lines in the query, or swap the `ORDER BY`
for one of the options in the comment next to it.

In [ ]:
n = 15  # <- change me: how many skills to show

skills = sql("""
SELECT a.repo_full_name AS repo, a.path, a.name, a.description, r.stars,
       a.sibling_count AS files, a.has_scripts, a.has_references,
       (SELECT COUNT(*) FROM artifacts c WHERE c.file_sha = a.file_sha) AS copies
FROM artifacts a
JOIN repos r ON r.full_name = a.repo_full_name
WHERE a.dedup_primary = 1                -- one row per distinct skill (keep this)
  -- AND a.has_scripts = 1               -- only skills that ship scripts
  -- AND r.language = 'Python'           -- only repos whose main language is Python
  -- AND a.name LIKE '%pdf%'             -- name contains "pdf"
ORDER BY r.stars DESC, a.repo_full_name, a.path   -- try: copies DESC, a.body_chars DESC, RANDOM()
LIMIT ?
""", (n,))
skills

## 4. View a skill

Prints the full SKILL.md text for one skill, i.e. the instructions an agent would read.

- **Easiest:** set `skill_index` to a row number from the table in section 3 (the number on the far left).
- **From a search:** after running section 5, uncomment the `matches.loc[...]` line and pick a row number there.
- **Any skill:** uncomment the last option and type a `repo` and `path` yourself.

Only use one option at a time (comment out the others). If you pick a copy, the text comes from its representative
and the output says which one. Section 6 shows the files for whatever skill you pick here, through the `skill`
variable.

In [ ]:
# Option 1: a row number from the section 3 table
skill_index = 0  # <- change me
repo, path = skills.loc[skill_index, ["repo", "path"]]

# Option 2: a row number from the section 5 search results (run section 5 first)
# repo, path = matches.loc[0, ["repo", "path"]]  # <- change the 0

# Option 3: any repo/path, typed directly (copies work too)
# repo, path = "obot-platform/obot", ".claude/skills/draft-release-blog/SKILL.md"

# Copies don't store text, so read it from the representative row with the same file_sha
skill = sql("""
SELECT rep.repo_full_name AS rep_repo, rep.path AS rep_path, rep.name, rep.content
FROM artifacts a
JOIN artifacts rep ON rep.file_sha = a.file_sha AND rep.dedup_primary = 1
WHERE a.repo_full_name = ? AND a.path = ?
""", (repo, path)).iloc[0]

print(f"{repo} / {path}")
if (skill.rep_repo, skill.rep_path) != (repo, path):
    print(f"(identical copy of {skill.rep_repo} / {skill.rep_path})")
print("-" * 80)
print(skill.content)

## 5. Keyword search

Finds distinct skills whose SKILL.md text contains `keyword`. Set it to anything you're curious about,
such as `"curl"`, `"password"`, `"subprocess"`, `"sudo"` or `".env"`, then re-run. The search ignores upper/lower case
for English letters and matches anywhere in the text, so `"key"` also matches `"monkey"`.

- The printed number is how many **distinct** SKILL.md files contain the keyword. Copies aren't counted again.
- `limit` only controls how many of those rows are shown, not how many are counted.
- To read one, go to section 4, use the `matches.loc[...]` option with that row number, and run it.

A match only means the text appears somewhere. Usually it's harmless: docs saying "set your API_KEY in .env",
an example command, a warning. It doesn't mean the skill is insecure, so read it before drawing conclusions.

In [ ]:
keyword = "API_KEY"  # <- change me: any text to look for
limit = 20           # <- change me: max rows to display

where = "dedup_primary = 1 AND content LIKE '%' || ? || '%'"
total = sql(f"SELECT COUNT(*) AS n FROM artifacts WHERE {where}", (keyword,))["n"][0]
print(f"{total} distinct SKILL.md files contain {keyword!r} (showing {min(total, limit)})")

matches = sql(f"""
SELECT repo_full_name AS repo, path, name, description
FROM artifacts
WHERE {where}
ORDER BY repo_full_name, path
LIMIT ?
""", (keyword, limit))
matches

## 6. Supporting files

A skill is a folder: the SKILL.md plus anything else the author added. That could be scripts the agent can run
(`scripts/setup.sh`), reference docs it reads when needed (`references/api.md`), templates or data files.
This cell lists the files next to the skill you picked in **section 4**. To check another skill, change it there
and re-run 4, then this cell.

- `file`: path inside the skill folder, `bytes`: file size
- `text_stored`: 1 if the dataset has the file's text. Binaries (images, etc.), very large files and folders with
  too many files aren't stored, so those show 0.

To read a file, paste a name from the `file` column into `show_file`, e.g. `show_file = "references/rule.md"`.
Many skills have no extra files at all, so an empty table is normal.

In [ ]:
# Uses `skill` from section 4
files = sql("""
SELECT entry_name AS file, entry_size AS bytes, content IS NOT NULL AS text_stored
FROM artifact_siblings
WHERE repo_full_name = ? AND artifact_path = ? AND entry_type = 'file'
ORDER BY entry_name
""", (skill.rep_repo, skill.rep_path))
print(f"{len(files)} files in {skill.rep_repo} / {skill.rep_path}")
display(files)

show_file = ""  # <- change me: a name from the "file" column; leave "" to skip
if show_file:
    text = sql("""
    SELECT content FROM artifact_siblings
    WHERE repo_full_name = ? AND artifact_path = ? AND entry_name = ?
    """, (skill.rep_repo, skill.rep_path, show_file))["content"]
    if text.empty:
        print(f"No file named {show_file!r} in this skill's folder (check the spelling, or the skill picked in section 4)")
    else:
        print(text[0] if text[0] is not None else "(text not stored for this file)")

## 7. Custom SQL

Write any SQLite query between the triple quotes and run the cell. The result shows up as a table.
The example lists the repos with the most distinct skills. Some other things to try:

```sql
-- first 10 rows of a table
SELECT * FROM repos LIMIT 10
```

```sql
-- distinct skills that ship scripts, from repos with 100+ stars
SELECT a.repo_full_name, a.path, a.name, r.stars
FROM artifacts a JOIN repos r ON r.full_name = a.repo_full_name
WHERE a.dedup_primary = 1 AND a.has_scripts = 1 AND r.stars >= 100
ORDER BY r.stars DESC
```

Tips: keep `dedup_primary = 1` when you want each unique skill counted once, and drop it to count every occurrence.
Section 2 shows a table's column names. Save results with `df = sql(query)` if you want to keep working with them in
section 8.

In [ ]:
# <- change me: replace the query with your own
query = """
SELECT r.full_name, r.stars, r.language, COUNT(*) AS distinct_skills
FROM artifacts a
JOIN repos r ON r.full_name = a.repo_full_name
WHERE a.dedup_primary = 1
GROUP BY r.full_name
ORDER BY distinct_skills DESC
LIMIT 20
"""
sql(query)

## 8. Playground

Free space for your own experiments: pandas on query results, quick checks, whatever. Everything from earlier
cells (`sql`, `skills`, `matches`, `skill`, `files`) is available here. Add more cells below if you need them
(hover under a cell and click **+ Code**).

In [ ]:
# Example (uncomment to try): how long are SKILL.md files, in characters?
# lengths = sql("SELECT name, body_chars FROM artifacts WHERE dedup_primary = 1")
# lengths["body_chars"].describe()
